In [0]:
bronze_df= spark.read.format("delta")\
    .load("/Volumes/workspace/bronzebike/bronzevolume/bronzedata/data")

display(bronze_df)    

In [0]:
# --- Notebook: 02_Refine_Silver ---

In [0]:
from pyspark.sql.functions import col, unix_timestamp, round, expr, when, to_timestamp



# 1. Cast coordinates to Double (Numbers)
bronze_df = bronze_df.withColumn("start_lat", col("start_lat").cast("double")) \
                     .withColumn("start_lng", col("start_lng").cast("double")) \
                     .withColumn("end_lat", col("end_lat").cast("double")) \
                     .withColumn("end_lng", col("end_lng").cast("double"))


bronze_df = bronze_df.withColumn("started_at", 
    to_timestamp(col("started_at"), "yyyy-MM-dd HH:mm:ss.SSS")
).withColumn("ended_at", 
    to_timestamp(col("ended_at"), "yyyy-MM-dd HH:mm:ss.SSS")
)                     

haversine_expr = """
    2 * 6371 * asin(sqrt(
        pow(sin(radians(end_lat - start_lat) / 2), 2) +
        cos(radians(start_lat)) * cos(radians(end_lat)) *
        pow(sin(radians(end_lng - start_lng) / 2), 2)
    ))
"""


silver_df = (bronze_df
    # 1. Clean NULLs and "False Starts"
    .filter(col("end_lat").isNotNull() & (col("start_station_id") != col("end_station_id")))
    .withColumn("duration_sec", unix_timestamp(col("ended_at")) - unix_timestamp(col("started_at")))
    .filter(col("duration_sec").between(60, 86400))

    # 2. Standardize Membership
    .withColumn("is_member", col("member_casual") == "member")

    # 3. Categorize Bike Type
    .withColumn("bike_category", 
        when(col("rideable_type").contains("electric"), "Electric")
        .otherwise("Manual"))

    # 4. Geospatial & Speed

    .withColumn("distance_km", round(expr(haversine_expr), 2))
    .withColumn("avg_speed_kmh", 
        when(col("duration_sec") > 0, 
             round(col("distance_km") / (col("duration_sec") / 3600), 2))
        .otherwise(0))


 )

silver_df=silver_df.drop("_rescued_data")
display(silver_df)

In [0]:
# --- Silver Layer: silver_trips (The "Fact" Stream) ---

silver_fact_df=(silver_df
    .withColumn("is_member", col("member_casual") == "member")
    .withColumn("bike_category", when(col("rideable_type").contains("electric"), "Electric").otherwise("Manual"))
    
    # D. DROP Descriptive Columns (They belong in the Dimension Table)
    .drop("start_station_name", "end_station_name", "member_casual", "rideable_type"))

# 2. Write to Silver Fact Table
silver_fact_df.write.format("delta").mode("append").saveAsTable("main.citibike_lakehouse.silver_trips")
## Why this split is correct:


In [0]:
# --- Silver Layer: dim_stations (SCD Type 2) ---
from pyspark.sql.functions import col, current_timestamp, lit, to_timestamp
raw_stations = silver_df\
    .select("start_station_id", "start_station_name", "start_lat", "start_lng") \
    .distinct() \
    .filter(col("start_station_id").isNotNull())

# 2. Prepare for Merge (Add metadata)
updates_df = raw_stations.withColumn("start_date", current_timestamp()) \
                         .withColumn("end_date", to_timestamp(lit("9999-12-31 23:59:59"))) \
                         .withColumn("is_current", lit(True))

# 3. SCD Type 2 MERGE Logic
# (Note: In a full pipeline, you would use a 'Union' approach for true SCD2, 
# but here is the standard Delta MERGE for station updates)
spark.sql("""
    MERGE INTO main.citibike_lakehouse.dim_stations AS target
    USING temp_updates AS source
    ON target.station_id = source.start_station_id AND target.is_current = true
    WHEN MATCHED AND (target.station_name <> source.start_station_name) THEN
      UPDATE SET is_current = false, end_date = current_timestamp()
    WHEN NOT MATCHED THEN
      INSERT (station_id, station_name, lat, lng, start_date, end_date, is_current)
      VALUES (source.start_station_id, source.start_station_name, source.start_lat, source.start_lng, current_timestamp(), '9999-12-31', true)
""")